# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All dataset components, such as record sets, fields, and columns, are referenced strictly by their `@id` identifiers as defined in the Croissant schema.

### Dataset Source
The dataset is hosted on the senscience FAIR^2 platform and described by a Croissant schema JSON-LD at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(metadata.name)
print(metadata.description)


## 2. Data Overview
Review available record sets, fields, and their IDs. This will help us identify which parts of the dataset to extract and analyze.

In [ ]:
# List all available record sets by their @id
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '(no name)')}")

# Show available fields and columns for each record set, by @id
for rs in record_sets:
    print(f"\nRecord Set '@id': {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"  Field @id: {field_id}")
        # Try to show columns for this field if available
        # These may be found in field['column'] (may be list or dict)
        field_obj = None
        # Try to get field object from the dataset.fields
        for f in dataset.fields:
            if f['@id'] == field_id:
                field_obj = f
                break
        if field_obj:
            columns = field_obj.get('column', [])
            if isinstance(columns, dict):
                columns = [columns]
            for col in columns:
                col_id = col['@id'] if isinstance(col, dict) else col
                print(f"    Column @id: {col_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets are referenced by their `@id`, as are individual fields/columns.

In [ ]:
# Choose one or more record sets by @id obtained from the overview above.
# For this dataset, there is likely a main record set containing the protein/peptide data.

record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")

# For analysis, pick the main protein abundance/analysis table. Choose the first non-empty dataframe.
main_record_set_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"\nSample columns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter on a numeric field, normalize, and group by a categorical field. All referenced fields/columns should be used via their `@id`.

In [ ]:
import numpy as np

# You should pick IDs from above, but as an example:
# Suppose our main table contains columns like: 'Accession', 'Description', 'Coverage', 'Peptide_count', 'MW', 'Sample1_abundance', etc.
# Let's attempt to find numeric fields and group fields
if main_record_set_id:
    df = dataframes[main_record_set_id]
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numerical_cols:
        # If numbers were loaded as strings, try to infer
        possible_numerical_cols = [c for c in df.columns if any(x in c.lower() for x in ['coverage', 'peptide', 'mw', 'abundance', 'count'])]
        for col in possible_numerical_cols:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    print(f"Numeric fields: {numerical_cols}")
    # Select a numeric field, default to Coverage if present
    chosen_numeric_field = None
    preferred = ['Coverage', 'Peptide_count', 'MW', 'Sample1_abundance', 'Abundance']
    for pref in preferred:
        for c in numerical_cols:
            if pref.lower() in c.lower():
                chosen_numeric_field = c
                break
        if chosen_numeric_field:
            break
    if not chosen_numeric_field and numerical_cols:
        chosen_numeric_field = numerical_cols[0]

    print(f"Using numeric field: {chosen_numeric_field}")
    threshold = df[chosen_numeric_field].mean() if chosen_numeric_field else 0
    print(f"Filtering to {chosen_numeric_field} > {threshold:.2f}")

    filtered_df = df[df[chosen_numeric_field] > threshold] if chosen_numeric_field else df
    print(filtered_df.head())

    # Normalize the field
    if chosen_numeric_field:
        filtered_df[f"{chosen_numeric_field}_normalized"] = (
            filtered_df[chosen_numeric_field] - filtered_df[chosen_numeric_field].mean()) / filtered_df[chosen_numeric_field].std()
        print(filtered_df[[chosen_numeric_field, f"{chosen_numeric_field}_normalized"]].head())

    # Try grouping by a categorical/textual field, e.g. 'Description'
    group_field = None
    preferred_group_fields = ['Description', 'Accession']
    for pref in preferred_group_fields:
        if pref in df.columns:
            group_field = pref
            break
    if group_field:
        grouped = filtered_df.groupby(group_field)[chosen_numeric_field].mean().sort_values(ascending=False)
        print(f"Grouped means by {group_field} (top 5):\n", grouped.head())
else:
    print("No main record set identified.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id and chosen_numeric_field:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,5))
    df[chosen_numeric_field].hist(bins=30)
    plt.title(f"Distribution of {chosen_numeric_field}")
    plt.xlabel(chosen_numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot of two numeric fields if available
    if len(numerical_cols) > 1:
        plt.figure(figsize=(7,5))
        plt.scatter(df[numerical_cols[0]], df[numerical_cols[1]], alpha=0.6)
        plt.xlabel(numerical_cols[0])
        plt.ylabel(numerical_cols[1])
        plt.title(f"{numerical_cols[0]} vs {numerical_cols[1]}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 mass spectrometry dataset using `mlcroissant`. The analysis included dynamic field selection and manipulation using only entity `@id` references, ensuring transparent and reproducible usage. Data was efficiently filtered, normalized, and visualized. Refer to the [FAIR^2 platform](https://sen.science/) for further metadata details and extended workflows.